# Weighted Lebesgue Spaces and Bessel-Sobolev Priors

This notebook demonstrates the new `WeightedLebesgue` space and how inner-product weighting affects
Laplacian-based priors via `BesselSobolevInverse` covariances.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from intervalinf import IntervalDomain, Function
from intervalinf.spaces import Lebesgue, WeightedLebesgue
from intervalinf.core.boundary import BoundaryConditions
from intervalinf.core.config import IntegrationConfig
from intervalinf.operators import Laplacian, BesselSobolevInverse

print("Imports successful!")

## 1. Create Plain and Weighted Lebesgue Spaces

In [ ]:
domain = IntervalDomain(0.1, 1.0)
bc = BoundaryConditions.dirichlet()
integration_config = IntegrationConfig(method="simpson", n_points=2000)

# Plain L² space
space_plain = Lebesgue(0, domain, basis=None, integration_config=integration_config)

# Weighted L²(r²) space
def weight_r2(r):
    return np.asarray(r, dtype=float) ** 2

space_weighted = WeightedLebesgue(0, domain, weight_r2, integration_config=integration_config)

print(f"Plain L² space created")
print(f"Weighted L²(r²) space created")

## 2. Compare Inner Products

In [ ]:
f = Function(domain, evaluate_callable=lambda r: np.sin(np.pi * np.asarray(r)))
g = Function(domain, evaluate_callable=lambda r: np.cos(2 * np.pi * np.asarray(r)))

inner_plain = space_plain.inner_product(f, g)
inner_weighted = space_weighted.inner_product(f, g)

print(f"⟨f, g⟩ (plain L²): {inner_plain:.6f}")
print(f"⟨f, g⟩_w (weighted): {inner_weighted:.6f}")
print(f"Difference: {abs(inner_plain - inner_weighted):.6f}")

## 3. Create Laplacians

In [ ]:
# Laplacian on plain L²
L_plain = Laplacian(space_plain, bc, 1.0, method="spectral", dofs=20, integration_config=integration_config)

# Laplacian on weighted L²(r²)
L_weighted = Laplacian(space_weighted, bc, 1.0, method="spectral", dofs=20, integration_config=integration_config)

print("Laplacians created.")
print(f"\nFirst 3 eigenvalues:")
for i in range(3):
    print(f"  λ_{i}: {L_plain.get_eigenvalue(i):.4f}")

## 4. Create Bessel-Sobolev Covariances

In [ ]:
k, s = 1.5, 1.0

C_plain = BesselSobolevInverse(domain=space_plain, codomain=space_plain, k=k, s=s, 
    L=L_plain, dofs=20, integration_config=integration_config)

C_weighted = BesselSobolevInverse(domain=space_weighted, codomain=space_weighted, k=k, s=s, 
    L=L_weighted, dofs=20, integration_config=integration_config)

print(f"Bessel-Sobolev C = (k² I - Δ)^{{-s}} created")
print(f"  k={k}, s={s}")
print(f"  Plain: uses fast transforms = {C_plain._can_use_fast_transforms}")
print(f"  Weighted: uses radial fast path = {C_weighted._radial_dirichlet_fast}")

## 5. Compare Prior Standard Deviations

In [ ]:
r_fine = np.linspace(0.1, 1.0, 300)

def estimate_std(C, L, r_eval, n_eig=15):
    var = np.zeros_like(r_eval)
    for j in range(n_eig):
        phi_j = L.get_eigenfunction(j)
        lambda_j = L.get_eigenvalue(j)
        scaling = (k**2 + lambda_j) ** (-s / 2.0)
        var += scaling * (phi_j(r_eval) ** 2)
    return np.sqrt(np.maximum(var, 0))

std_plain = estimate_std(C_plain, L_plain, r_fine)
std_weighted = estimate_std(C_weighted, L_weighted, r_fine)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(r_fine, std_plain, 'b-', linewidth=2, label='Plain L²')
axes[0].plot(r_fine, std_weighted, 'r--', linewidth=2, label='Weighted L²(r²)')
axes[0].set_xlabel('r')
axes[0].set_ylabel('Std Dev')
axes[0].set_title('Prior Standard Deviation')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

ratio = std_weighted / np.maximum(std_plain, 1e-12)
axes[1].plot(r_fine, ratio, 'g-', linewidth=2)
axes[1].set_xlabel('r')
axes[1].set_ylabel('Ratio (weighted / plain)')
axes[1].set_title('Effect of Weighting')
axes[1].axhline(y=1.0, color='k', linestyle=':', alpha=0.5)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nStandard deviation:")
print(f"  @ r=0.1: plain={std_plain[0]:.4f}, weighted={std_weighted[0]:.4f}")
print(f"  @ r=1.0: plain={std_plain[-1]:.4f}, weighted={std_weighted[-1]:.4f}")

## Summary

✅ **WeightedLebesgue** implements $L^2(w)$ as a unified mechanism via `MassWeightedHilbertSpace`

✅ **Weighted priors** automatically encode geometric structure (e.g., spherical shells with $w(r)=r^2$)

✅ **Transparent to operators** — the same `BesselSobolevInverse` code handles both plain and weighted domains

✅ **Self-adjoint** on the space's inner product, whether plain or weighted